# Phase 4: SVM Integration with ThunderSVM (Kaggle)
**CSC14120 - Parallel Programming**

## ThunderSVM vs LIBSVM
| Feature | LIBSVM | ThunderSVM |
|:--------|:-------|:-----------|
| Platform | CPU | GPU (CUDA) |
| Speed | Baseline | 10-100x faster |
| Algorithm | SMO | GPU-parallel SMO |
| Accuracy | Reference | Same (identical algorithm) |

## Objectives
- Load trained autoencoder weights from Phase 3
- Extract 8192-dimensional features using encoder
- Train SVM classifier with **ThunderSVM** (RBF kernel, C=10, gamma=auto)
- Evaluate on test set and generate confusion matrix
- Target accuracy: 60-65%

## Pipeline
1. **Feature Extraction**: Encoder forward pass on all 60K images (GPU)
2. **SVM Training**: ThunderSVM on GPU - much faster than LIBSVM
3. **Evaluation**: Predict on 10K test features, compute accuracy

In [ ]:
# Check GPU
!nvidia-smi
!nvcc --version

In [ ]:
# Copy project from Kaggle dataset (auto-unzipped)
import os
import glob
import shutil

# Find project directory in Kaggle input (already unzipped)
input_dir = '/kaggle/input'
src_dirs = glob.glob(f'{input_dir}/**/src', recursive=True)

if src_dirs:
    project_dir = os.path.dirname(src_dirs[0])
    print(f"Found project at: {project_dir}")
    shutil.copytree(project_dir, '/kaggle/working/project', dirs_exist_ok=True)
    
    # Change to project directory
    for root, dirs, _ in os.walk('/kaggle/working/project'):
        if 'src' in dirs:
            os.chdir(root)
            break
else:
    print("ERROR: Project not found in /kaggle/input")
    print("Please add your project as a Kaggle dataset")

print(f"Working directory: {os.getcwd()}")
!ls

In [ ]:
# Download CIFAR-10 dataset
import urllib.request
import tarfile

os.makedirs('data', exist_ok=True)

if not os.path.exists('data/data_batch_1.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve(
        'https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz', 
        'data/cifar.tar.gz'
    )
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/

print('CIFAR-10 ready!')

In [ ]:
# Use sklearn SVM (more stable than ThunderSVM)
# ThunderSVM often crashes due to CUDA version incompatibility

print("Using sklearn SVM (stable, CPU-based)")
print("Note: We'll use stratified sampling to reduce training time")

from sklearn.svm import SVC
print("sklearn SVM ready!")

In [ ]:
# Find and copy Phase 3 weights
weights_file = 'phase3_opt.weights'

if not os.path.exists(weights_file):
    # Search in Kaggle input directories
    weight_files = glob.glob('/kaggle/input/**/*phase3*.weights', recursive=True)
    weight_files += glob.glob('/kaggle/input/**/*.weights', recursive=True)
    
    if weight_files:
        src_weights = weight_files[0]
        print(f"Found weights: {src_weights}")
        shutil.copy(src_weights, weights_file)
        print(f"Copied to: {weights_file}")
    else:
        print("ERROR: No weights file found!")
        print("Please upload phase3_opt.weights to your Kaggle dataset")
else:
    print(f"Found weights file: {weights_file}")

!ls -lh {weights_file}

In [ ]:
# Build feature extractor
print("Installing LIBSVM...")
!git clone --depth 1 https://github.com/cjlin1/libsvm.git libsvm_src 2>/dev/null || echo "LIBSVM already cloned"
!cd libsvm_src && make lib

# Copy LIBSVM files to project
!cp libsvm_src/svm.h include/ 2>/dev/null || echo "svm.h already exists"
!cp libsvm_src/svm.cpp src/ 2>/dev/null || echo "svm.cpp already exists"

# Check required files exist
print("\nChecking required files...")
!ls -la src/main_phase4.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/layers_gpu_opt.cu src/dataset.cpp src/svm_wrapper.cpp

# Build (Kaggle uses sm_70 for P100 or sm_75 for T4)
print("\nBuilding feature_extractor...")
!nvcc -O3 -std=c++17 -arch=sm_70 --expt-relaxed-constexpr \
    --use_fast_math -Xptxas -O3 --maxrregcount=64 \
    -DUSE_OPTIMIZED_KERNELS -DWITH_SVM -DWITH_LIBSVM \
    -Iinclude -Ilibsvm_src -lcublas -lcudnn \
    -o feature_extractor \
    src/main_phase4.cu src/layers_gpu.cu src/gpu_autoencoder.cu \
    src/layers_gpu_opt.cu src/dataset.cpp src/svm_wrapper.cpp libsvm_src/svm.cpp

# Verify build
!if [ -f feature_extractor ]; then echo "Build SUCCESS!"; ls -lh feature_extractor; else echo "Build FAILED!"; fi

In [ ]:
# Extract features ONLY (skip LIBSVM training) using --extract-only flag
import time

print("Extracting features with GPU autoencoder...")
print("Using --extract-only to skip slow LIBSVM training")
start = time.time()

# --extract-only: extracts all features and saves to .bin files, then exits
# No SVM training happens - we'll use ThunderSVM in Python
!./feature_extractor --data data --weights phase3_opt.weights --extract-only

feat_time = time.time() - start
print(f"\nFeature extraction completed in {feat_time:.2f}s")

# Verify files were created
!ls -lh *.bin 2>/dev/null || echo "No .bin files found"

In [ ]:
# Load extracted features
import numpy as np

feature_dim = 8192

if os.path.exists('train_features.bin') and os.path.exists('test_features.bin'):
    print("Loading pre-extracted features...")
    
    # Load training features
    train_features = np.fromfile('train_features.bin', dtype=np.float32)
    num_train = len(train_features) // feature_dim
    train_features = train_features.reshape(num_train, feature_dim)
    
    # Load test features  
    test_features = np.fromfile('test_features.bin', dtype=np.float32)
    num_test = len(test_features) // feature_dim
    test_features = test_features.reshape(num_test, feature_dim)
    
    print(f"Train features: {train_features.shape}")
    print(f"Test features: {test_features.shape}")
else:
    print("ERROR: Feature files not found!")
    !ls -la *.bin 2>/dev/null || echo "No .bin files"

In [ ]:
# Load CIFAR-10 labels
import struct

def load_cifar10_labels(data_dir):
    train_labels = []
    test_labels = []
    
    # Load training labels (5 batches)
    for i in range(1, 6):
        batch_file = f"{data_dir}/data_batch_{i}.bin"
        with open(batch_file, 'rb') as f:
            for _ in range(10000):
                label = struct.unpack('B', f.read(1))[0]
                train_labels.append(label)
                f.read(3072)  # Skip image data
    
    # Load test labels
    with open(f"{data_dir}/test_batch.bin", 'rb') as f:
        for _ in range(10000):
            label = struct.unpack('B', f.read(1))[0]
            test_labels.append(label)
            f.read(3072)  # Skip image data
    
    return np.array(train_labels), np.array(test_labels)

train_labels, test_labels = load_cifar10_labels('data')
print(f"Train labels: {len(train_labels)}")
print(f"Test labels: {len(test_labels)}")

# Trim labels to match features
train_labels = train_labels[:num_train]
test_labels = test_labels[:num_test]

In [ ]:
# L2 normalize features AND optimize memory
from sklearn.preprocessing import normalize
import gc

print("L2 normalizing features...")

# Overwrite variables to save RAM
train_features = normalize(train_features, norm='l2')
test_features = normalize(test_features, norm='l2')

# Rename for compatibility, but point to same memory
train_features_norm = train_features
test_features_norm = test_features

print("Normalization done.")
gc.collect()

In [ ]:
# Train SVM with stratified sampling & Aggressive Memory Cleanup
import time
from sklearn.svm import SVC
import numpy as np
import gc

# 1. Stratified sampling
samples_per_class = 1000
num_classes = 10

print(f"Stratified sampling: {samples_per_class} per class = {samples_per_class * num_classes} total")

train_indices = []
for c in range(num_classes):
    class_indices = np.where(train_labels == c)[0]
    selected = class_indices[:samples_per_class]
    train_indices.extend(selected)

train_indices = np.array(train_indices)

# 2. Create Training Set
X_train = train_features_norm[train_indices]
y_train = train_labels[train_indices]

print(f"Training set created: {X_train.shape}")

# 3. CRITICAL: Delete the huge 50k arrays to free RAM
del train_features
del train_features_norm
del train_labels
gc.collect()
print("Freed memory of full dataset. Ready to train.")

# SVM parameters (same as project spec)
C = 10.0
gamma = 1.0 / X_train.shape[1]  # auto = 1/dim

print(f"\nSVM Parameters:")
print(f"  Kernel: RBF | C: {C} | gamma: {gamma:.6e}")
print(f"  Training samples: {len(X_train)}")

# 4. Train
# cache_size=1000MB is safer
svm = SVC(kernel='rbf', C=C, gamma=gamma, cache_size=1000)

print(f"\nTraining SVM (Estimated: 5-10 mins)...")
start = time.time()
svm.fit(X_train, y_train)
train_time = time.time() - start
print(f"SVM training completed in {train_time:.2f}s")
print(f"Support vectors: {svm.n_support_.sum()}")

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
start = time.time()
predictions = svm.predict(test_features_norm)
eval_time = time.time() - start

# Calculate accuracy
accuracy = np.mean(predictions == test_labels)

print(f"\n" + "="*60)
print("PHASE 4 RESULTS (ThunderSVM)")
print("="*60)
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(f"Training samples: {len(train_labels)}")
print(f"Test samples: {len(test_labels)}")
print(f"SVM Training time: {train_time:.2f}s")
print(f"Evaluation time: {eval_time:.2f}s")
print("="*60)

# Target check
if 0.60 <= accuracy <= 0.65:
    print("TARGET MET! (60-65%)")
elif accuracy > 0.65:
    print("ABOVE TARGET! Excellent!")
else:
    print(f"Below target (expected 60-65%, got {accuracy*100:.2f}%)")

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Compute confusion matrix
cm = confusion_matrix(test_labels, predictions)

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix - ThunderSVM\nAccuracy: {accuracy*100:.2f}%')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig('phase4_thundersvm_confusion.png', dpi=150)
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(test_labels, predictions, target_names=class_names))

In [ ]:
# Per-class accuracy
print("\nPer-class Accuracy:")
for i, name in enumerate(class_names):
    mask = test_labels == i
    class_acc = np.mean(predictions[mask] == test_labels[mask])
    print(f"  {name:12s}: {class_acc*100:.2f}%")

In [ ]:
# Save results
import pandas as pd

results = {
    'metric': ['test_accuracy', 'train_samples', 'test_samples', 'feature_dim',
               'svm_c', 'svm_gamma', 'svm_training_time', 'evaluation_time',
               'svm_library'],
    'value': [accuracy, len(train_labels), len(test_labels), train_features_norm.shape[1],
              C, gamma, train_time, eval_time,
              'ThunderSVM' if use_thundersvm else 'sklearn']
}

df = pd.DataFrame(results)
df.to_csv('phase4_thundersvm_results.csv', index=False)
print("Results saved!")

# Copy to Kaggle output
shutil.copy('phase4_thundersvm_results.csv', '/kaggle/working/')
shutil.copy('phase4_thundersvm_confusion.png', '/kaggle/working/')

print("\nOutput files:")
!ls -lh /kaggle/working/*.csv /kaggle/working/*.png 2>/dev/null

## Performance Comparison

### ThunderSVM vs LIBSVM

| Metric | LIBSVM (CPU) | ThunderSVM (GPU) |
|:-------|:-------------|:-----------------|
| Training Time | 30-60+ minutes | 1-5 minutes |
| Speedup | 1x | 10-100x |
| Accuracy | ~60-65% | ~60-65% (same) |

### Why ThunderSVM is faster:
- GPU-parallel SMO algorithm
- Efficient kernel computation on GPU
- Same mathematical algorithm, just parallelized

### Notes:
- Accuracy should be identical (same algorithm)
- Minor differences possible due to floating-point precision